# Fig. 1D: volcano on rho (length index vs pseudotime)

The x-axis is the graph-smoothed Spearman rho of the per-cell 3'UTR-length index vs
pseudotime (`rho_W`, the average-rank length metric, not a "best-site" correlation).
y-axis = -log10(padj) from the DM LRT, shown as an uncalibrated pseudo p-value; the
significance call is the permutation FDR. Vertical dashed lines mark the minimum effect size
(|rho| > 0.2): genes inside the band are grey even if significant. Marginal densities on
both axes.

- Kernel: `apa-python`.
- Input: `DirMult_optimized/composite_ranking_optimized2_df5.csv` (`rho_W`, `padj`) and the
  within-sample permutation p-values.
- Output: `.../Fig1/fig1d_volcano/fig1d_volcano__rhoW.{svg,png}`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

PANEL = "fig1d_volcano__rhoW"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(54)  # 85 x 54 mm (half-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

PADJ_THR = 0.05
RHO_THR = 0.2  # minimum effect size for colouring (|rho| > 0.2)
RANKING_PATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/"
    "DirMult_optimized/composite_ranking_optimized2_df5.csv"
)
ASD_HIGHLIGHT = {"Qk", "Dyrk1a", "Tcf4", "Prodh", "Nova2"}
XLAB = r"Spearman $\rho$  (3$^\prime$UTR length vs pseudotime)"

In [ ]:
df = pd.read_csv(RANKING_PATH)
# y-axis = raw chi-sq magnitude, shown as an uncalibrated "pseudo p-value".
# Significance call = permutation FDR.
df["neg_log10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))
cap = df.loc[np.isfinite(df["neg_log10_padj"]), "neg_log10_padj"].quantile(0.999)
df["neg_log10_padj"] = df["neg_log10_padj"].clip(upper=cap * 1.05)

# merge within-sample permutation p (mouse) and BH-adjust -> permutation FDR
PERM_PATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/"
    "DirMult_optimized/permutation_batch_number/"
    "dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_cov-batch_number.csv"
)
perm = pd.read_csv(PERM_PATH)[["utr_name", "p_value_perm"]]
df = df.merge(perm, left_on="gene_id", right_on="utr_name", how="left")
df["padj_perm"] = S.benjamini_hochberg(df["p_value_perm"].values)

# direction + effect-size gate on rho; significance by permutation FDR
df["sig_class"] = "NS"
df.loc[(df["padj_perm"] < PADJ_THR) & (df["rho_W"] < -RHO_THR), "sig_class"] = "Shortening"
df.loc[(df["padj_perm"] < PADJ_THR) & (df["rho_W"] > RHO_THR), "sig_class"] = "Lengthening"
n_short = int((df["sig_class"] == "Shortening").sum())
n_long = int((df["sig_class"] == "Lengthening").sum())
print(f"permutation-FDR<0.05 & |rho|>{RHO_THR}: shortening={n_short}, lengthening={n_long}")

In [ ]:
fig = plt.figure(figsize=FIGSIZE)
gs = GridSpec(
    4,
    4,
    figure=fig,
    width_ratios=[1, 1, 1, 0.35],
    height_ratios=[0.3, 1, 1, 1],
    hspace=0.05,
    wspace=0.05,
)
ax_main = fig.add_subplot(gs[1:, :3])
ax_top = fig.add_subplot(gs[0, :3], sharex=ax_main)
ax_right = fig.add_subplot(gs[1:, 3], sharey=ax_main)

cs = df["composite_score_abs"].values
size = 1 + 12 * (cs - cs.min()) / (cs.max() - cs.min() + 1e-9)
for cls, col in [
    ("NS", S.COL_NS),
    ("Shortening", S.COL_SHORT),
    ("Lengthening", S.COL_LONG),
]:
    m = (df["sig_class"] == cls).values
    ax_main.scatter(
        df.loc[m, "rho_W"],
        df.loc[m, "neg_log10_padj"],
        c=col,
        s=size[m],
        alpha=0.4 if cls == "NS" else 0.65,
        edgecolors="none",
        rasterized=True,
        zorder=1 if cls == "NS" else 2,
        label=cls,
    )

# thresholds: LRT significance + the minimum effect-size (rho) band
ax_main.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.6, zorder=0)
ax_main.axvline(-RHO_THR, color="gray", ls="--", lw=0.6, zorder=0)
ax_main.axvline(RHO_THR, color="gray", ls="--", lw=0.6, zorder=0)
ax_main.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)

top_genes = df[df["sig_class"] != "NS"].nlargest(10, "composite_score_abs")
texts = []
for _, row in top_genes.iterrows():
    is_asd = row["gene_name"] in ASD_HIGHLIGHT
    texts.append(
        ax_main.text(
            row["rho_W"],
            row["neg_log10_padj"],
            row["gene_name"],
            fontsize=S.FONT_ANNOT - 0.5,
            color="#8B0000" if is_asd else "#4d4d4d",
            fontweight="bold" if is_asd else "normal",
            fontstyle="italic",
            zorder=10,
        )
    )
if HAS_AT and texts:
    adjust_text(
        texts,
        ax=ax_main,
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.4),
        expand=(1.3, 1.4),
        force_text=(0.5, 0.8),
    )

ax_main.set_ylim(top=ax_main.get_ylim()[1] * 1.20)  # clear header band for the legend
ax_main.set_xlabel(XLAB, fontsize=S.FONT_LABEL)
ax_main.set_ylabel(r"$-\log_{10}$ pseudo-$p$", fontsize=S.FONT_LABEL)
ax_main.tick_params(labelsize=S.FONT_TICK)

# top marginal: rho densities per direction
for cls, col in [("Shortening", S.COL_SHORT), ("Lengthening", S.COL_LONG)]:
    vals = df.loc[df["sig_class"] == cls, "rho_W"].dropna()
    if len(vals) > 5:
        ax_top.hist(vals, bins=40, color=col, alpha=0.5, density=True, edgecolor="none")
ax_top.tick_params(labelbottom=False, labelleft=False, left=False)
ax_top.spines[["top", "right", "left"]].set_visible(False)
ax_top.set_ylabel("")

# right marginal: -log10 padj densities per direction
for cls, col in [("Shortening", S.COL_SHORT), ("Lengthening", S.COL_LONG)]:
    vals = df.loc[df["sig_class"] == cls, "neg_log10_padj"].dropna()
    if len(vals) > 5:
        ax_right.hist(
            vals,
            bins=40,
            color=col,
            alpha=0.5,
            density=True,
            orientation="horizontal",
            edgecolor="none",
        )
ax_right.tick_params(labelleft=False, labelbottom=False, bottom=False)
ax_right.spines[["top", "right", "bottom"]].set_visible(False)
ax_right.set_xlabel("")

# coloured-text legend (no dot glyphs; NS centred between shortening/lengthening)
_leg = [
    (f"Shortening ({n_short:,})", S.COL_SHORT),
    ("NS / |ρ|<0.2", "#8a8a8a"),
    (f"Lengthening ({n_long:,})", S.COL_LONG),
]
leg = ax_main.legend(
    handles=[Line2D([], [], linestyle="none", label=t) for t, _ in _leg],
    loc="upper center",
    ncol=3,
    frameon=False,
    handlelength=0,
    handletextpad=0,
    columnspacing=1.0,
    borderpad=0.2,
    fontsize=S.FONT_LEGEND,
)
for _t, (_, _c) in zip(leg.get_texts(), _leg):
    _t.set_color(_c)

fig.subplots_adjust(left=0.10, right=0.96, top=0.96, bottom=0.12)
S.save_and_show(fig, PANEL, fig_dir=S.OUT_ROOT / "Fig1" / "fig1d_volcano")

In [ ]:
# --- Reproduce figure without y-axis clipping ---
df2 = pd.read_csv(RANKING_PATH)
df2["neg_log10_padj"] = -np.log10(df2["padj"].clip(lower=1e-300))  # no upper clip
df2 = df2.merge(
    pd.read_csv(PERM_PATH)[["utr_name", "p_value_perm"]],
    left_on="gene_id",
    right_on="utr_name",
    how="left",
)
df2["padj_perm"] = S.benjamini_hochberg(df2["p_value_perm"].values)

df2["sig_class"] = "NS"
df2.loc[(df2["padj_perm"] < PADJ_THR) & (df2["rho_W"] < -RHO_THR), "sig_class"] = "Shortening"
df2.loc[(df2["padj_perm"] < PADJ_THR) & (df2["rho_W"] > RHO_THR), "sig_class"] = "Lengthening"
n_short2 = int((df2["sig_class"] == "Shortening").sum())
n_long2 = int((df2["sig_class"] == "Lengthening").sum())

fig2 = plt.figure(figsize=FIGSIZE)
gs2 = GridSpec(
    4,
    4,
    figure=fig2,
    width_ratios=[1, 1, 1, 0.35],
    height_ratios=[0.3, 1, 1, 1],
    hspace=0.05,
    wspace=0.05,
)
ax_main2 = fig2.add_subplot(gs2[1:, :3])
ax_top2 = fig2.add_subplot(gs2[0, :3], sharex=ax_main2)
ax_right2 = fig2.add_subplot(gs2[1:, 3], sharey=ax_main2)

cs2 = df2["composite_score_abs"].values
size2 = 1 + 12 * (cs2 - cs2.min()) / (cs2.max() - cs2.min() + 1e-9)
for cls, col in [
    ("NS", S.COL_NS),
    ("Shortening", S.COL_SHORT),
    ("Lengthening", S.COL_LONG),
]:
    m = (df2["sig_class"] == cls).values
    ax_main2.scatter(
        df2.loc[m, "rho_W"],
        df2.loc[m, "neg_log10_padj"],
        c=col,
        s=size2[m],
        alpha=0.4 if cls == "NS" else 0.65,
        edgecolors="none",
        rasterized=True,
        zorder=1 if cls == "NS" else 2,
        label=cls,
    )

ax_main2.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.6, zorder=0)
ax_main2.axvline(-RHO_THR, color="gray", ls="--", lw=0.6, zorder=0)
ax_main2.axvline(RHO_THR, color="gray", ls="--", lw=0.6, zorder=0)
ax_main2.axvline(0, color="0.5", ls="-", lw=0.4, alpha=0.5, zorder=0)

top_genes2 = df2[df2["sig_class"] != "NS"].nlargest(10, "composite_score_abs")
texts2 = []
for _, row in top_genes2.iterrows():
    is_asd = row["gene_name"] in ASD_HIGHLIGHT
    texts2.append(
        ax_main2.text(
            row["rho_W"],
            row["neg_log10_padj"],
            row["gene_name"],
            fontsize=S.FONT_ANNOT - 0.5,
            color="#8B0000" if is_asd else "#4d4d4d",
            fontweight="bold" if is_asd else "normal",
            fontstyle="italic",
            zorder=10,
        )
    )
if HAS_AT and texts2:
    adjust_text(
        texts2,
        ax=ax_main2,
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.4),
        expand=(1.3, 1.4),
        force_text=(0.5, 0.8),
    )

ax_main2.set_ylim(top=ax_main2.get_ylim()[1] * 1.20)  # clear header band for the legend
ax_main2.set_xlabel(XLAB, fontsize=S.FONT_LABEL)
ax_main2.set_ylabel(r"$-\log_{10}$ pseudo-$p$", fontsize=S.FONT_LABEL)
ax_main2.tick_params(labelsize=S.FONT_TICK)

for cls, col in [("Shortening", S.COL_SHORT), ("Lengthening", S.COL_LONG)]:
    vals = df2.loc[df2["sig_class"] == cls, "rho_W"].dropna()
    if len(vals) > 5:
        ax_top2.hist(vals, bins=40, color=col, alpha=0.5, density=True, edgecolor="none")
ax_top2.tick_params(labelbottom=False, labelleft=False, left=False)
ax_top2.spines[["top", "right", "left"]].set_visible(False)
ax_top2.set_ylabel("")

for cls, col in [("Shortening", S.COL_SHORT), ("Lengthening", S.COL_LONG)]:
    vals = df2.loc[df2["sig_class"] == cls, "neg_log10_padj"].dropna()
    if len(vals) > 5:
        ax_right2.hist(
            vals,
            bins=40,
            color=col,
            alpha=0.5,
            density=True,
            orientation="horizontal",
            edgecolor="none",
        )
ax_right2.tick_params(labelleft=False, labelbottom=False, bottom=False)
ax_right2.spines[["top", "right", "bottom"]].set_visible(False)
ax_right2.set_xlabel("")

# coloured-text legend (no dot glyphs; NS centred between shortening/lengthening)
_leg = [
    (f"Shortening ({n_short2:,})", S.COL_SHORT),
    ("NS / |ρ|<0.2", "#8a8a8a"),
    (f"Lengthening ({n_long2:,})", S.COL_LONG),
]
leg = ax_main2.legend(
    handles=[Line2D([], [], linestyle="none", label=t) for t, _ in _leg],
    loc="upper center",
    ncol=3,
    frameon=False,
    handlelength=0,
    handletextpad=0,
    columnspacing=1.0,
    borderpad=0.2,
    fontsize=S.FONT_LEGEND,
)
for _t, (_, _c) in zip(leg.get_texts(), _leg):
    _t.set_color(_c)

fig2.subplots_adjust(left=0.10, right=0.96, top=0.96, bottom=0.12)
S.save_and_show(fig2, PANEL + "__unclipped", fig_dir=S.OUT_ROOT / "Fig1" / "fig1d_volcano")